# Model training

## The shape contract

`sequence_to_tensor` returns **one sample. There is no batch axis in it.**

| call | returns | reading | pairs with |
| --- | --- | --- | --- |
| `sequence_to_tensor(series_dir)` | `(24, 224, 224, 1)` | `(K, H, W, C)` - the 24 slices are a *depth* axis, 1 channel because the images are monochrome | `Conv3D` |
| `sequence_to_tensor(series_dir, as_channels=True)` | `(224, 224, 24)` | `(H, W, K)` - the 24 slices are *channels* | `Conv2D` |

The batch axis is added by **stacking samples**, so a batch of `B` series is
`(B, 24, 224, 224, 1)` or `(B, 224, 224, 24)`. A batch of one is `x[None, ...]`,
i.e. `(1, 24, 224, 224, 1)` - that is where the leading `1` comes from, and it
belongs to the data, never to `layers.Input(shape=...)`.

`layers.Input(shape=...)` takes the **per-sample** shape; Keras adds the batch
axis itself. So `Input(shape=(1, 24, 224, 224, 1))` declares a rank-6 tensor
`(None, 1, 24, 224, 224, 1)`, while `Conv3D` needs rank 5 `(B, d, h, w, c)`.
That was the bug. Below, the shape is never hardcoded at all - it is read off
the data with `x.shape[1:]`, which drops the batch axis and leaves exactly the
per-sample shape.

Labels are separate: the twelve `soft_*` columns of `data/meta/train_index.csv`,
one row per series, so `(B, 12)` against the model's `(B, 12)` output.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, models

# functions/ is two levels up from notebooks/david/
sys.path.append(str(Path.cwd().parents[1] / "functions"))
from sequence_to_tensor import sequence_to_tensor, study_to_tensor  # noqa: E402

DATA = Path("../../data")

LABELS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
          "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
SOFT = [f"soft_{name}" for name in LABELS]

## Building the model

The layout decides the convolution, so the builder reads the rank of one sample
and picks:

* rank 4 `(K, H, W, 1)` -> a **3D CNN**, the slices are a depth axis the kernels
  move through, so the model can see across slices.
* rank 3 `(H, W, K)` -> a **2.5D CNN**, the slices are channels, so every kernel
  already sees all 24 at once but the model has no notion of slice order.

Two things that were in the first draft are gone:

* **`TimeDistributed` is dropped.** It exists to apply a layer at every step of a
  *time* axis, and there is no time axis here. Wrapped around the pooling it made
  Keras read the 24 slices as time and hand a rank-4 slice to a 3D pool, and
  wrapped around the last `Dense` it produced `(B, T, 12)` against labels of
  `(B, 12)`.
* **the duplicated `Dense(20)`.** Both lines read `shared_feature`, so the second
  overwrote the first and one layer was silently dropped. They are chained now.

In [ ]:
def build_model(input_shape, n_labels=len(LABELS)):
    """One sample's shape -> a compiled-ready model.

    input_shape is the shape of ONE series, with no batch axis:
        (24, 224, 224, 1) -> 3D CNN     (sequence_to_tensor default)
        (224, 224, 24)    -> 2.5D CNN   (as_channels=True)
    """
    if len(input_shape) == 4:
        conv, pool, gap = layers.Conv3D, layers.MaxPooling3D, layers.GlobalAveragePooling3D
        # (1, 2, 2) first: pool the two image axes but keep all 24 slices for now.
        pool_sizes = [(1, 2, 2), (2, 2, 2), (2, 2, 2)]
    elif len(input_shape) == 3:
        conv, pool, gap = layers.Conv2D, layers.MaxPooling2D, layers.GlobalAveragePooling2D
        pool_sizes = [(2, 2), (2, 2), (2, 2)]
    else:
        raise ValueError(
            f"one sample is (K, H, W, 1) or (H, W, K), got {input_shape}. "
            "If this has a leading 1, you passed a batch of one instead of a sample."
        )

    inputs = layers.Input(shape=input_shape)

    net = inputs
    for filters, pool_size in zip((32, 64, 64), pool_sizes):
        net = conv(filters, 3, padding="same", activation="relu")(net)
        net = pool(pool_size=pool_size, padding="same")(net)

    # Global pooling, not Flatten: it collapses every spatial axis to one number
    # per filter, so the head does not depend on the image size.
    net = gap()(net)

    net = layers.Dense(20, activation="relu")(net)
    net = layers.Dense(20, activation="relu")(net)

    # 12 independent binary decisions, so sigmoid (not softmax): a knee can carry
    # several findings at once.
    outputs = layers.Dense(n_labels, activation="sigmoid")(net)

    return models.Model(inputs=inputs, outputs=outputs)

In [ ]:
def model_training_25d(x: np.ndarray, labels: np.ndarray, epochs: int = 1, **fit_kwargs):
    """Train on a batch of series.

        x       (B, 24, 224, 224, 1)  or  (B, 224, 224, 24)
        labels  (B, 12)               floats in [0, 1] (the soft_* columns)
    """
    if len(x) != len(labels):
        raise ValueError(f"{len(x)} samples but {len(labels)} label rows")

    # The one line that used to be hardcoded: the per-sample shape comes from the
    # data, so the model can never disagree with what sequence_to_tensor produced.
    model = build_model(x.shape[1:], n_labels=labels.shape[1])

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.AUC(name="auc"),
            tf.keras.metrics.Recall(name="recall"),
            tf.keras.metrics.Precision(name="precision"),
        ],
    )

    model.fit(x, labels, epochs=epochs, **fit_kwargs)
    return model

## Smoke test on random data

Two patients, so `B = 2`. Note how each patient is built at the *sample* shape and
the batch axis appears only from `np.stack`.

In [ ]:
# Sequence layout, what sequence_to_tensor(series_dir) gives you.
p1_x = np.random.rand(24, 224, 224, 1).astype("float32")
p2_x = np.random.rand(24, 224, 224, 1).astype("float32")

p1_y = np.array([1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0], dtype="float32")  # 12 findings
p2_y = np.array([0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0], dtype="float32")

X_train = np.stack([p1_x, p2_x])   # (2, 24, 224, 224, 1)
Y_train = np.stack([p1_y, p2_y])   # (2, 12)

print("X", X_train.shape, " one sample", X_train.shape[1:])
print("Y", Y_train.shape)

trained_model = model_training_25d(X_train, Y_train)
trained_model.summary()

In [ ]:
# The same data in the 2.5D layout, what as_channels=True gives you.
# Same builder, same trainer - only the shape changed.
X_channels = np.transpose(X_train[..., 0], (0, 2, 3, 1))   # (2, 224, 224, 24)

print("X", X_channels.shape, " one sample", X_channels.shape[1:])
model_25d = model_training_25d(X_channels, Y_train)
print("output", model_25d.output_shape, "against labels", Y_train.shape)

## Real data

`sequence_to_tensor` reads DICOMs from `data/train_series/<study>/<series>/`, which
is not in this checkout. The mirror is: `data/volumes/<series>.npz` holds `data` of
shape `(24, 224, 224)` **uint8** - the same slices, missing only the channel axis
and the normalisation. `[..., None] / 255` restores the exact contract, so the two
paths are interchangeable and training on the mirror will not mismatch inference
through `sequence_to_tensor`.

In [ ]:
index = pd.read_csv(DATA / "meta" / "train_index.csv")   # one row per series
print(len(index), "series,", index.npz.nunique(), "volumes")

batch = index.head(8)   # keep it small, this runs on CPU

X = np.stack([np.load(DATA / "volumes" / name)["data"] for name in batch.npz])
X = (X[..., None].astype("float32") / 255.0)   # (B, 24, 224, 224, 1), values in [0, 1]
Y = batch[SOFT].to_numpy("float32")            # (B, 12)

print("X", X.shape, X.dtype, f"[{X.min():.2f}, {X.max():.2f}]")
print("Y", Y.shape, "e.g.", dict(zip(LABELS, Y[0])))

In [ ]:
model = model_training_25d(X, Y, epochs=1, batch_size=2)

### Straight from `sequence_to_tensor`

The same batch, built from DICOMs instead of the mirror. This needs
`data/train_series/` to be present.

In [ ]:
# series_df = pd.read_csv(DATA / "train_series.csv")
#
# uids = index.head(8).StudyInstanceUID
# X = np.stack([study_to_tensor(uid, series_df, data_root=DATA, axis="X") for uid in uids])
# print(X.shape)        # (8, 24, 224, 224, 1) - stacking is what adds the batch axis
#
# # ...or the 2.5D layout, note there is no trailing 1 in this one:
# X = np.stack([study_to_tensor(uid, series_df, data_root=DATA, axis="X",
#                               as_channels=True) for uid in uids])
# print(X.shape)        # (8, 224, 224, 24)